# Layers & the Forward Pass

This notebook accompanies the **ML Viz** lesson on layers and the forward pass.
We'll write the forward pass in matrix form with NumPy, reproduce the lesson's
2→2→1 worked example exactly, and visualize what hidden width does to the
functions a network can express.

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/neural-networks/03-layers-and-forward-pass

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Intuition — a network is stacked matrix multiplies

A neural network's **forward pass** is almost embarrassingly simple: each layer is one matrix
multiply plus a bias plus an element-wise activation, `h = f(Wx + b)`, and a deep network just
repeats that rule. The matrix `W` holds one neuron per row; the activation `f` bends the result so
that stacking layers actually adds power (without it, the whole stack collapses to a single matrix).
**Width** (neurons per layer) and **depth** (number of layers) are the two dials: width adds folds
within a layer, depth composes folds on top of folds. This notebook builds the forward pass from
scratch, visualizes what width and depth do to the function, and checks it against `jax`.

## The layer equation

A layer of $m$ neurons reading $n$ inputs is one matrix multiplication:

$$\mathbf{h} = f\left(W\mathbf{x} + \mathbf{b}\right), \qquad W \in \mathbb{R}^{m \times n},\; \mathbf{b} \in \mathbb{R}^m$$

- Each **row** of $W$ is one neuron's weight vector (rows = outputs, columns = inputs)
- The activation $f$ is applied **element-wise**
- A deep network is just this rule applied repeatedly, layer after layer

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# The 2 -> 2 -> 1 network from the lesson:
W1 = np.array([[2.0, -1.0],     # hidden neuron 1 (row 1)
               [0.5,  1.0]])    # hidden neuron 2 (row 2)
b1 = np.array([0.5, -3.0])
W2 = np.array([[2.0, 3.0]])     # output neuron weights the 2 hidden activations
b2 = np.array([1.0])

x = np.array([1.0, 2.0])

# Step 1 - hidden pre-activations (each row of W1 dots with x)
z1 = W1 @ x + b1
print('z1 =', z1)                       # [ 0.5 -0.5]

# Step 2 - ReLU element-wise: neuron 2 is clamped to 0 (inactive)
h1 = np.maximum(0, z1)
print('h1 =', h1)                       # [0.5 0. ]

# Step 3 - output layer + sigmoid
z2 = W2 @ h1 + b2
y = sigmoid(z2)
print('z2 =', z2)                       # [2.]
print('y  =', y.round(4))               # [0.8808] -> 'class 1' with p ~ 0.88

**What to notice:** trace the three steps — the pre-activations `z1 = [0.5, −0.5]`, then ReLU
**clamps neuron 2 to 0** (it's inactive for this input), then the output sigmoid gives `p ≈ 0.88`.
Each row of `W1` is one neuron dotting with the input; the whole layer is a single matrix multiply.

## A reusable forward pass

The same three lines generalize to any number of layers — and to whole
**batches**: stack $B$ examples as the rows of $X \in \mathbb{R}^{B \times n}$ and
compute $H = f(XW^\top + \mathbf{b})$. Nothing else changes.

In [ ]:
def init_network(sizes, seed=42):
    """Deterministic weights for a layer-size sequence, e.g. [2, 3, 1]."""
    rng = np.random.RandomState(seed)
    return [(rng.randn(m, n) * np.sqrt(2 / n), np.zeros(m))
            for n, m in zip(sizes[:-1], sizes[1:])]

def forward(X, params, act=np.tanh):
    """Forward pass for a batch X of shape (B, n). Output layer is linear."""
    H = X
    for i, (W, b) in enumerate(params):
        Z = H @ W.T + b
        H = Z if i == len(params) - 1 else act(Z)   # no activation on output
    return H

params = init_network([2, 3, 3, 1])
X_batch = np.array([[1.0, 2.0],
                    [0.0, 0.0],
                    [-1.5, 0.5]])

print('layer shapes:', [W.shape for W, _ in params])   # [(3,2), (3,3), (1,3)]
print('batch output:\n', forward(X_batch, params).round(4))
n_params = sum(W.size + b.size for W, b in params)
print('total parameters:', n_params)

**What to notice:** the same `forward` function handles any depth and a whole **batch** at once —
stacking the three inputs as rows of `X` gives three outputs in one matmul, no loop over examples.
The printed layer shapes `(3,2)→(3,3)→(1,3)` and parameter count follow directly from the layer
sizes.

## What hidden width does

Each hidden ReLU neuron contributes one "fold line" to the function the
network expresses. Few neurons → a coarse, piecewise-linear surface; many
neurons → a richly curved one. Let's draw the output surface of random
2 → h → 1 ReLU networks for increasing width $h$.

In [ ]:
relu = lambda z: np.maximum(0, z)

xx, yy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
fig.suptitle('Output surface of a random 2 → h → 1 ReLU network', color='white', y=1.02)

for ax, h in zip(axes, [2, 8, 64]):
    params = init_network([2, h, 1], seed=7)
    Z = forward(grid, params, act=relu).reshape(xx.shape)
    im = ax.contourf(xx, yy, Z, levels=24, cmap='RdBu_r')
    ax.set_title(f'h = {h}  ({4 * h + 1} params)', color='white', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

**What to notice:** each ReLU neuron adds one "fold line" to the surface. With `h=2` the output
is coarse and blocky; by `h=64` it's a richly curved landscape. **Width = capacity**: more hidden
units → more folds → more expressive functions (at the cost of more parameters).

## Depth composes features

Width adds folds *within* one layer; depth lets later layers fold the
*already-folded* surface again. With the same parameter budget, a deeper
network produces visibly more intricate structure.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
fig.suptitle('Same idea, deeper: random tanh networks', color='white', y=1.02)

architectures = [[2, 8, 1], [2, 8, 8, 1], [2, 8, 8, 8, 1]]
for ax, sizes in zip(axes, architectures):
    params = init_network(sizes, seed=3)
    Z = forward(grid, params, act=np.tanh).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=24, cmap='RdBu_r')
    n = sum(W.size + b.size for W, b in params)
    ax.set_title(' → '.join(map(str, sizes)) + f'  ({n} params)', color='white', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

**What to notice:** at a comparable parameter budget, the **deeper** networks produce visibly
more intricate structure than the shallow one — because each layer folds the *already-folded* output
of the previous one. Depth composes features, which is why deep-and-narrow often beats shallow-and-wide
for the same parameter count.

## The library way — validate the forward pass against `jax`

The forward pass is pure linear algebra, so any framework computes it identically given the same
weights. The cell reimplements `forward` with `jax.numpy` and asserts the outputs match our NumPy
version bit-for-bit — the same computation `torch.nn.Linear` / `flax` perform under the hood.

In [ ]:
import jax.numpy as jnp

def forward_jax(X, params, act=jnp.tanh):
    H = jnp.asarray(X)
    for i, (W, b) in enumerate(params):
        Z = H @ jnp.asarray(W).T + jnp.asarray(b)
        H = Z if i == len(params) - 1 else act(Z)
    return H

params = init_network([2, 5, 4, 1])
out_np  = forward(X_batch, params)
out_jax = np.array(forward_jax(X_batch, params))
print('numpy output:\n', out_np.round(5))
print('jax   output:\n', out_jax.round(5))
assert np.allclose(out_np, out_jax, atol=1e-6), "numpy and jax forward passes must match"
print('\nour from-scratch forward pass == jax ✓')

**What to notice:** identical outputs — the forward pass is framework-agnostic linear algebra.
A `torch.nn.Sequential` of `Linear`+activation layers does exactly this; the framework only adds
autodiff and GPU execution on top of the same `Wx + b` chain.

## Gotchas & tradeoffs

- **Shape conventions bite.** Rows of `W` = outputs, columns = inputs; a batch is `X @ W.T + b`.
  Getting `W` vs `W.T` wrong is the most common forward-pass bug (and it may not error, just give
  wrong shapes).
- **No activation on the output layer** for regression/logits — applying one (e.g. squashing logits
  before a softmax loss) is a classic mistake.
- **Parameters scale with width × depth.** A layer `n→m` has `mn + m` parameters; wide layers
  dominate the count, which is why big models are often deep-and-narrow.
- **Deeper isn't free.** More layers → more matmuls per step and harder-to-train gradients
  (vanishing/exploding) — the motivation for the normalization and init tricks in later lessons.

In [ ]:
# Parameter count scales with width x depth
for sizes in [[784, 128, 10], [784, 256, 256, 10], [784, 64, 64, 64, 64, 10]]:
    n = sum(m * k + m for k, m in zip(sizes[:-1], sizes[1:]))
    print(f'{str(sizes):<28} -> {n:>7,} parameters')

**What to notice:** the single wide hidden layer (`784→128`) already costs ~100k parameters, and
most of a network's weights live in its widest layers. Counting parameters this way — `mn + m` per
layer — is how you budget a model's size and memory.

## Key takeaways

- A layer is one matrix multiplication plus an element-wise activation: $\mathbf{h} = f(W\mathbf{x} + \mathbf{b})$, with $mn + m$ parameters for $n \to m$.
- The forward pass alternates matrix products and activations; batches come for free by stacking inputs as rows.
- ReLU clamps negative pre-activations to zero — inactive neurons contribute nothing for that input.
- **Width** adds capacity within a layer; **depth** composes features, which is far more parameter-efficient.
- Without the non-linearity between layers, any stack of matrices collapses to a single linear map.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — A dense layer

One layer is one affine map plus a nonlinearity, applied to a whole **batch** at once:

$$H = \sigma(XW + \mathbf{b}), \qquad X \in \mathbb{R}^{n \times d_{in}}, \; W \in \mathbb{R}^{d_{in} \times d_{out}}$$

Implement it with ReLU as the default activation. The checks verify the output shape, that ReLU never outputs negatives, and that the identity activation recovers the raw affine map.

In [ ]:
def relu(z):
    return np.maximum(z, 0.0)


def dense(X, W, b, activation=relu):
    """One dense layer: activation(X @ W + b)."""
    X = np.asarray(X, dtype=float)

    # TODO(you): the affine map X @ W + b, passed through the activation
    return ...

In [ ]:
# Checks — run me
rng = np.random.default_rng(0)
X = rng.standard_normal((4, 3))
W = rng.standard_normal((3, 5))
b = rng.standard_normal(5)

out = dense(X, W, b)
assert out.shape == (4, 5), "batch of 4, width 5 -> output (4, 5)"
assert np.all(out >= 0), "ReLU output is never negative"

ident = dense(X, W, b, activation=lambda z: z)
assert np.allclose(ident, X @ W + b), "identity activation returns the raw affine map"
assert np.allclose(relu(ident), out), "dense = activation(X @ W + b)"

# Edge case: a batch of size 1 must still come back 2-D (1, d_out), not squeeze
# to a 1-D vector -- shape consistency matters once this feeds another layer.
out_single = dense(X[:1], W, b)
assert out_single.shape == (1, 5), "a single-example batch keeps its leading batch axis"

# Edge case: an all-negative pre-activation is entirely clipped to zero by ReLU
W_neg = -np.abs(rng.standard_normal((3, 5))) - 1.0   # guarantee X @ W_neg + b_neg < 0
out_neg = dense(np.abs(X) + 1.0, W_neg, -np.abs(b) - 100.0)
assert np.all(out_neg == 0.0), "a layer that is negative everywhere is entirely dead under ReLU"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def dense(X, W, b, activation=relu):
    X = np.asarray(X, dtype=float)
    return activation(X @ W + b)
```

</details>

### Exercise 2 — Counting parameters

Each layer from width $d_{in}$ to $d_{out}$ costs $d_{in} \times d_{out}$ weights plus $d_{out}$ biases. Walk consecutive pairs of the size list and add it up. The classic MNIST MLP $[784, 128, 10]$ should come out to **101,770** — and note from the last check how parameter cost shapes architecture choices.

In [ ]:
def count_params(sizes):
    """Total trainable parameters of an MLP with the given layer sizes."""
    total = 0
    for n_in, n_out in zip(sizes[:-1], sizes[1:]):
        # TODO(you): weights (n_in * n_out) plus biases (n_out)
        total += ...
    return total

In [ ]:
# Checks — run me
assert count_params([2, 3, 1]) == 13, "(2*3 + 3) + (3*1 + 1) = 13"
assert count_params([784, 128, 10]) == 101770, "the classic MNIST MLP"
assert count_params([5, 7]) == 42, "a single layer: 5*7 + 7"

deep = count_params([10, 100, 100, 1])
wide = count_params([10, 250, 1])
assert deep > wide, "two 100-wide layers cost more than one 250-wide layer here"

# Edge case: a single-number list has no layers at all -- 0 parameters, not an error
assert count_params([42]) == 0, "no layer transitions -> no weights or biases"

# Edge case: a width-1 bottleneck still costs exactly in_dim*1 + 1 for that layer
assert count_params([100, 1, 100]) == (100 * 1 + 1) + (1 * 100 + 100), "narrow bottleneck layer"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def count_params(sizes):
    total = 0
    for n_in, n_out in zip(sizes[:-1], sizes[1:]):
        total += n_in * n_out + n_out
    return total
```

</details>

---
## 🔬 Extra practice — from Open-Deep-ML

Two more drills on the forward pass: a numerically-stable log-softmax (the
form you'd actually use inside a cross-entropy loss), and a stateful `Dense`
layer class -- the same affine map as Exercise 1, but as an object that owns
its weights and hands the *update rule* off to an injected optimizer, which is
how real training frameworks (Keras, PyTorch `nn.Module`) structure a layer.

- [`39` implementation-of-log-softmax-function](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/39_implementation-of-log-softmax-function)
- [`40` implementing-a-custom-dense-layer-in-python](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/40_implementing-a-custom-dense-layer-in-python)

### Exercise 3 — log-softmax (DML #39)

`log(softmax(x))` shows up constantly (cross-entropy loss is defined in terms
of it), and computing softmax first, then taking its log, is a numerical trap:
softmax's normalizer can vanish to 0 or blow up before you ever get to the
log. The stable form subtracts the max *once* and folds the rest into a single
`log-sum-exp`.

In [ ]:
def log_softmax(scores):
    """DML #39: log(softmax(scores)), computed stably in one pass."""
    scores = np.asarray(scores, dtype=float)
    shifted = scores - np.max(scores)   # subtract max for numerical stability
    # TODO(you): log_sum_exp = log(sum(exp(shifted))); return shifted - log_sum_exp
    return ...

In [ ]:
# Checks — run me (DML's own published test cases, from tests.json)
assert np.allclose(log_softmax([1, 2, 3]), [-2.4076, -1.4076, -0.4076], atol=1e-4)
assert np.allclose(log_softmax([1, 1, 1]), [-1.0986, -1.0986, -1.0986], atol=1e-4)
assert np.allclose(log_softmax([1, 1, 0.0000001]), [-0.862, -0.862, -1.862], atol=1e-3)

# Exponentiating must recover an actual probability distribution
probs = np.exp(log_softmax([1, 2, 3]))
assert abs(probs.sum() - 1.0) < 1e-9, "exp(log_softmax(x)) is softmax(x), which sums to 1"

# Numerical stability: huge scores must not overflow, and log_softmax is
# shift-invariant -- adding the same constant to every score changes nothing
huge = log_softmax([1000, 1001, 1002])
assert np.all(np.isfinite(huge)), "large scores must not overflow to inf/nan"
assert np.allclose(huge, log_softmax([0, 1, 2]), atol=1e-9), "log_softmax(x + c) == log_softmax(x)"
print("✅ DML 39 log-softmax passed")

<details>
<summary>💡 Show solution</summary>

```python
def log_softmax(scores):
    scores = np.asarray(scores, dtype=float)
    shifted = scores - np.max(scores)
    log_sum_exp = np.log(np.sum(np.exp(shifted)))
    return shifted - log_sum_exp
```

</details>

### Exercise 4 — a stateful Dense layer with an injected optimizer (DML #40)

Exercise 1 implemented a dense layer as a pure function; DML #40 asks for the
same affine map as a **class** that owns its own weights and delegates *how*
they get updated to an injected optimizer object -- exactly the `step(w, g,
state)` optimizers from the gradient-descent notebook, wrapped in a full
forward/backward layer interface. Implement `forward_pass` (remember the
input -- `backward_pass` needs it) and `backward_pass` (parameter gradients,
an optimizer-driven update, and the gradient handed back to the previous
layer).

In [ ]:
class MockOptimizer:
    """The simplest possible optimizer: a fixed-learning-rate gradient step."""
    def __init__(self, lr=0.1):
        self.lr = lr

    def update(self, weights, grad):
        return weights - self.lr * grad


class Dense:
    """DML #40: a fully-connected layer that owns its weights and delegates
    updates to an injected optimizer (one instance per parameter tensor)."""

    def __init__(self, n_units, input_shape):
        self.n_units = n_units
        self.input_shape = input_shape
        self.W = None
        self.w0 = None
        self.layer_input = None

    def initialize(self, optimizer):
        limit = 1 / np.sqrt(self.input_shape[0])
        rng = np.random.default_rng(0)
        self.W = rng.uniform(-limit, limit, (self.input_shape[0], self.n_units))
        self.w0 = np.zeros((1, self.n_units))
        self.W_opt = optimizer
        self.w0_opt = copy.deepcopy(optimizer)

    def parameters(self):
        return self.W.size + self.w0.size

    def forward_pass(self, X):
        self.layer_input = X
        # TODO(you): the affine map X @ W + w0 (layer_input is already saved above --
        # backward_pass needs it)
        return ...

    def backward_pass(self, accum_grad):
        W = self.W    # the *old* weights -- needed below for the gradient to the previous layer

        # TODO(you): parameter gradients. grad_W = layer_input^T @ accum_grad;
        # grad_w0 = column-sums of accum_grad (keepdims=True so it stays (1, n_units))
        grad_W = ...
        grad_w0 = ...

        # Delegate the actual update rule to the injected optimizers
        self.W = self.W_opt.update(self.W, grad_W)
        self.w0 = self.w0_opt.update(self.w0, grad_w0)

        # TODO(you): gradient flowing to the previous layer: accum_grad @ W^T (the OLD W above)
        return ...

    def output_shape(self):
        return (self.n_units,)

In [ ]:
# Checks — run me
import copy

layer = Dense(n_units=2, input_shape=(3,))
layer.initialize(MockOptimizer(lr=0.1))

# Overwrite the (randomly-initialized) weights with known values so the test
# is hand-checkable and doesn't depend on the RNG stream.
layer.W = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
layer.w0 = np.array([[0.5, -0.5]])

X = np.array([[1.0, 2.0, 3.0]])
out = layer.forward_pass(X)
assert np.allclose(out, [[4.5, 4.5]]), "X @ W + w0, worked by hand: [1+0+3, 0+2+3] + [0.5, -0.5]"
assert layer.parameters() == 8, "3*2 weights + 2 biases"

accum_grad = np.array([[1.0, 1.0]])
grad_to_prev = layer.backward_pass(accum_grad)
assert np.allclose(grad_to_prev, [[1.0, 1.0, 2.0]]), "accum_grad @ W^T, using the OLD W"
assert np.allclose(layer.W, [[0.9, -0.1], [-0.2, 0.8], [0.7, 0.7]]), "W updated via the injected optimizer"
assert np.allclose(layer.w0, [[0.4, -0.6]]), "w0 updated via its own optimizer instance"

# Edge case: a single-neuron output layer (n_units=1) with a batch of inputs,
# and lr=0 so the optimizer update is a true no-op -- easy to verify by hand.
layer_single = Dense(n_units=1, input_shape=(2,))
layer_single.initialize(MockOptimizer(lr=0.0))
layer_single.W = np.array([[2.0], [-1.0]])
layer_single.w0 = np.array([[0.0]])
X_batch = np.array([[1.0, 1.0], [3.0, 0.0]])
out_batch = layer_single.forward_pass(X_batch)
assert np.allclose(out_batch, [[1.0], [6.0]]), "batch forward: [2*1-1*1, 2*3-1*0]"
_ = layer_single.backward_pass(np.array([[1.0], [1.0]]))
assert np.allclose(layer_single.W, [[2.0], [-1.0]]), "lr=0 -- the optimizer update is a true no-op"
print("✅ DML 40 custom-dense-layer passed")

<details>
<summary>💡 Show solution</summary>

```python
class Dense:
    def __init__(self, n_units, input_shape):
        self.n_units = n_units
        self.input_shape = input_shape
        self.W = None
        self.w0 = None
        self.layer_input = None

    def initialize(self, optimizer):
        limit = 1 / np.sqrt(self.input_shape[0])
        rng = np.random.default_rng(0)
        self.W = rng.uniform(-limit, limit, (self.input_shape[0], self.n_units))
        self.w0 = np.zeros((1, self.n_units))
        self.W_opt = optimizer
        self.w0_opt = copy.deepcopy(optimizer)

    def parameters(self):
        return self.W.size + self.w0.size

    def forward_pass(self, X):
        self.layer_input = X
        return X @ self.W + self.w0

    def backward_pass(self, accum_grad):
        W = self.W
        grad_W = self.layer_input.T @ accum_grad
        grad_w0 = np.sum(accum_grad, axis=0, keepdims=True)
        self.W = self.W_opt.update(self.W, grad_W)
        self.w0 = self.w0_opt.update(self.w0, grad_w0)
        return accum_grad @ W.T

    def output_shape(self):
        return (self.n_units,)
```

</details>